# Frequent Itemset Mining & Association Rules

## Association Rules Discovery 

> 关联规则挖掘

物品与购物车模型(item and baskets):
- 商品与购物袋
- 词与文档
- 碱基对与基因
- 药物与病人

First: 定义
- Frequent itemsets
- Assocaition rules

    confidence support interestingness

Then: Algorithms for finding frequent itemsets
- Finding frequent pairs
- A-Priori algorithm
- PCY algo


## 问题的定义

- Itemset : 物品集合
- Frequent Itemset: 频繁集

目标: 找到经常一起出现的集合

支持度(support): 数据集对一组频繁集的支持度

> {Beer,Bread} = 2 表示全体Baskets中 {Beer,Bread}出现的频数是 2  

在给定的 support threshold $s$ (阈值)下,至少出现s次的物品组被称之为`频繁集`

上述概念的一个样例

`Items` = {milk, coke, pepsi, beer, juice}  
`Support threshold` = 3 baskets  
> B1 = {m,c,b}, B1 = {m,c,b}  
> B3 = {m,b},  B4 = {c,j}  
> B5 = {m,p,b}, B6 = {m,c,b,j}  
> B7 = {c,b,j}, B8 = {b,c}  


`Frequent itemsets` = {m},{c}{b},{j},{m,b}



## 关联规则

IF-THEN规则
$$
\{i_1,i_2,...,i_k\}\to j
$$
表示如果一个basket包含全部的$i_1,i_2,...,i_k$,那么他有可能去包含j

置信度$Confidence$
$$
conf(I\to j) = \frac{support(I\cup j)}{support(I)}
$$
高置信的关联规则不一定会Interesing

Interet of an association rule $I\to j$
$$
Interest(I\to j) = |conf(I\to j)-Pr[j]| 
$$
其中$Pr[j]$表示j出现的概率

沿用上面的例子,对于关联规则${m,b}\to c$
- supprot = 2
- confidence = 2/4 = 0.5
- Interest = |0.5-5/8| = 1/8

## 关联规则挖掘

问题的形式化:
    
Find all association rules with support $\ge$ s and confidence $\ge$ c

问题的挑战: 找到`frequent itemsets`

问题求解:

Step1: Find all frequent itemset I 

Step2: Rule generation

对于I的每一个规则子集A,$A\to I|A$:
- 只要I是FI(frequent itemset),那么A也是frequent的
- 只要一轮就可以计算规则的confidence,$confidence(A,B-\to C,D)=support(A,B,C,D)/support(A,B)$
- 如果$A,B,C\to D$小于confidence 那么 $A,B\to C,D$也小于,原因分子一样,分母变大 

关联规则挖掘的步骤:
- 计算全部的frequent itemset(根据s)
- 生成规则(根据c)


## 如何发现 Frequent itemsets

存储方式: 磁盘文件  
关键资源: 内存

最困难的问题: 发现 frequent pairs of items$\{i_1,i_2\}$

方案:
- 生成全部的 itemsets
- 最后只会计数最终frequeent的itemsets, how?

朴素的方法: 
- 使用双层嵌套生成items对 n(n-1)/2
- 如果有100K的items,大约需要20G的内存

如何实现pair的count计算:
- 使用矩阵,三角矩阵,4 bytes per pair 计算全部
- 使用一张数据表 $[i,j,c]$ 12 bytes pre pair 计算出现

## 数据过多无法载入内存怎么办?

> A-Priori Algorithm,基于优先级的优化算法

一个两次遍历的算法`A-Priori`降低了内存需求,相比与原始的设计

关键想法: 单调性,如果 I 出现了 s 次, 那么作为J的子集,J 至少出现了 s 次

pass1: 读取每一个baskets并且在内存中计算每一个独立item出现的次数,挖掘次数大于s的items

pass2: 第二次遍历只关注 第一次发现的 items 的组合的次数

## Triple是如何获取的?

由二元组join形成,但是不一定有效$\{b,c\}+\{c,j\}\to \{b,c,j\}$不一定有效,因为$\{b,j\}$未知

> A-Priori是可以计算全部 Frequent Itemsets的

## PCY算法

> 对A-Priori提升

pass1 of PCY:
    
- 除了计数,维护一个hash table包含很多buckets适合内存,桶机制
- 将pair hash 到一个桶中,在内存桶中加一
- 如果桶的 total count 小于s, 那么桶中所有的pair都不符合要求

Passs2 计算仅仅能够 hash 到 frequent buckets 中的pairs

所以PCY算法更多的是进行一次计算上的优化,等于添加了一个锁文件

## 小于两次迭代的方法

使用`采样`来模拟统计,降低s以提高发现frequent的概率